# The Sample Variance and the Famous n − 1

Estimating a population's variance from a sample seems easy: average the squared deviations, done. But the obvious formula — plugging the *sample* mean where the population mean belongs — is **biased**: it systematically *underestimates* the true variance. The fix is tiny and famous: divide by $n - 1$ instead of $n$, giving the standard sample variance $s^2$. This notebook derives the naive estimator, exposes its bias with a fully-enumerated paper-in-a-hat game, and shows the correction working exactly. It is based on the fourth video of Lesson 5 (Sampling and Point Estimation).

## 1. Introduction

Variance measures spread — the average squared deviation from the mean. Two five-person height datasets, both with mean 160 cm, make the idea visible: one with the dots huddled together (small variance), one with them scattered from 151 to 170 (large variance).

You learned the *population* formula in Week 2. But in statistics you usually have only a **sample**: no $\mu$, no $N$. How do you estimate the population variance from a sample alone? The answer takes a detour through a subtle trap — and this notebook walks straight into it, on purpose, before fixing it.

After completing this notebook you will be able to:

- Derive the naive estimator $\widehat{\mathrm{Var}}(X) = \frac1n \sum (x_i - \bar{x})^2$ from the sample-mean machinery.
- Demonstrate its **bias** by exhaustive enumeration (the 1-2-3 hat game).
- State and use the unbiased **sample variance** $s^2 = \frac{1}{n-1} \sum (x_i - \bar{x})^2$.
- Judge when the $n$ vs. $n-1$ distinction matters — and when it signals deeper problems.

## 2. Theory

### The population variance, recalled

$$
\mathrm{Var}(X) = \sigma^2 = \frac{1}{N} \sum_{i=1}^{N} (x_i - \mu)^2
$$

— the average squared deviation from the population mean.

### Building an estimator, step by step

Variance is itself an expectation, so the sample-mean techniques should apply. Define a new random variable

$$
Y = (X - \mu)^2
$$

Then the population variance is just $E[Y] = \mu_Y = \frac1N \sum y_i$ — a **population mean of $Y$**. And population means we know how to estimate: with sample means! For $n$ samples,

$$
\bar{y} = \frac1n \sum_{i=1}^n y_i
\qquad\Longrightarrow\qquad
\widehat{\mathrm{Var}}(X) = \frac1n \sum_{i=1}^n (x_i - \mu)^2
$$

(the hat marks an estimate: the population formula with $N$ replaced by $n$).

One problem remains: the expression still contains **$\mu$**, which you don't know either. So — let's *cheat* — replace it with the sample mean $\bar{x}$:

$$
\widehat{\mathrm{Var}}(X) = \frac1n \sum_{i=1}^n (x_i - \bar{x})^2
$$

Every ingredient is now computable from the sample, and it seems intuitively right. Will the cheat go unpunished?

### The cheat gets caught: bias

It turns out substituting $\bar{x}$ for $\mu$ introduces an error that makes the formula **biased** — a statistics term meaning it systematically over- or under-estimates its target. This one *underestimates* the population variance. (Intuition: $\bar{x}$ is fitted to the sample, sitting closer to these particular points than $\mu$ does, so squared deviations from $\bar{x}$ come out too small.)

### The exhibit: the 1-2-3 hat game

Three sheets of paper — 1, 2, 3 — in a hat; draw one, score its number, replace it. As a random variable: $\mu = 2$ and

$$
\sigma^2 = \frac{(-1)^2 + 0^2 + 1^2}{3} = \frac{2}{3}
$$

Play the game **twice** (with replacement): $n = 2$. There are only **9 equally likely outcomes**, so we can enumerate every possible sample, compute each one's naive variance estimate, and average: the result is $\frac13$ — but the target is $\frac23$. The naive estimator is biased low, caught red-handed by exhaustive enumeration.

### The fix: divide by $n - 1$

Adjust the divisor from $n$ to $n - 1$:

$$
s^2 = \frac{1}{n-1} \sum_{i=1}^n (x_i - \bar{x})^2
$$

($s$ because it plays the role of $\sigma$.) Rerunning the enumeration with $s^2$: the average over all 9 samples is exactly $\frac23$ — **unbiased**. We won't rigorously prove it here; just know that dividing by $n-1$ makes the sample variance unbiased in general.

### Perspective

- As $n$ grows, the difference fades: $\div 3$ vs $\div 2$ is large; $\div 1000$ vs $\div 999$ is negligible. Practitioner's rule: *if $n$ vs $n-1$ visibly changes your answer, your real problem is a small sample.*
- Some accepted techniques *do* divide by $n$ — notably **maximum likelihood estimation** (coming in a later lesson). But $s^2$ with $n - 1$ is the most common estimator of variance, and the default in the rest of this course (and in `np.var(x, ddof=1)` / `pandas`' default).

## 3. Worked Examples

### Example 1: the two height datasets

Dataset A (huddled): $158, 159, 160.5, 161, 161.5$ — mean 160. Squared deviations: $4, 1, 0.25, 1, 2.25$; sum $8.5$:

$$
\widehat{\mathrm{Var}} = \frac{8.5}{5} = 1.7 \qquad s^2 = \frac{8.5}{4} = 2.125
$$

Dataset B (spread): $151, 154, 159, 166, 170$ — mean 160. Squared deviations: $81, 36, 1, 36, 100$; sum $254$:

$$
\widehat{\mathrm{Var}} = \frac{254}{5} = 50.8 \qquad s^2 = \frac{254}{4} = 63.5
$$

(Quick sanity check from the video: B's points sit roughly 7 from the mean, and $7^2 = 49 \approx 50.8$.) In both datasets the $s^2$ estimate is slightly larger — dividing by $n-1$ always inflates, exactly countering the cheat's shrinkage.

### Example 2: the hat game, one sample worked out

Sample $(1, 3)$: $\bar{x} = 2$; naive: $\frac{(1-2)^2 + (3-2)^2}{2} = 1$; corrected: $\frac{2}{1} = 2$. Sample $(2, 2)$: both estimators give 0 — no sample of identical values can show spread, which is part of why small samples underestimate.

### Example 3: when the divisor matters

$n = 3$, sum of squared deviations $= 12$: naive 4 vs $s^2 = 6$ — a 50% difference. $n = 1000$, same average spread: the two estimates differ by 0.1%. The correction is essential in principle, invisible at scale.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

dataset_a = np.array([158, 159, 160.5, 161, 161.5])
dataset_b = np.array([151, 154, 159, 166, 170])

for name, d in [("A (huddled)", dataset_a), ("B (spread) ", dataset_b)]:
    naive = np.var(d)                # ddof=0: divide by n
    s2 = np.var(d, ddof=1)           # ddof=1: divide by n-1
    print(f"dataset {name}: mean = {d.mean():.0f}, "
          f"naive (1/n) = {naive:.3f}, s^2 (1/(n-1)) = {s2:.3f}")

### The hat game: enumerate all 9 samples and expose the bias

The video's table, computed: every possible two-draw sample, its mean, and both variance estimates — then the averages that convict and acquit.

In [ ]:
values = [1, 2, 3]
rows = []
for x1 in values:
    for x2 in values:
        m = (x1 + x2) / 2
        ssd = (x1 - m)**2 + (x2 - m)**2
        rows.append({"sample": f"({x1}, {x2})", "mean": m,
                     "naive (÷n)": ssd / 2, "s² (÷(n-1))": ssd / 1})

table = pd.DataFrame(rows)
print(table.to_string(index=False))

sigma2 = np.var(values)
print(f"\npopulation variance:            {sigma2:.4f}  (= 2/3)")
print(f"average of naive estimates:     {table['naive (÷n)'].mean():.4f}  (= 1/3 -- biased low!)")
print(f"average of s² estimates:        {table['s² (÷(n-1))'].mean():.4f}  (= 2/3 -- unbiased)")

### The bias holds for realistic populations too

Sample repeatedly from a large population and average each estimator: the naive one undershoots by exactly the factor $\frac{n-1}{n}$; $s^2$ hits the target.

In [ ]:
rng = np.random.default_rng(0)

population = rng.normal(160, 8, size=100_000)
sigma2 = population.var()
print(f"population variance sigma^2 = {sigma2:.2f}\n")
print(f"{'n':>4} {'avg naive':>10} {'avg s^2':>9} {'sigma^2 (n-1)/n':>16}")
for n in [2, 3, 5, 10, 30]:
    samples = population[rng.integers(0, len(population), size=(40_000, n))]
    naive = samples.var(axis=1, ddof=0).mean()
    s2 = samples.var(axis=1, ddof=1).mean()
    print(f"{n:>4} {naive:>10.2f} {s2:>9.2f} {sigma2 * (n - 1) / n:>16.2f}")

### Why the cheat shrinks the deviations

For any sample, the sum of squared deviations is *smallest* around the sample's own mean — so measuring from $\bar{x}$ instead of $\mu$ can only reduce it.

In [ ]:
rng = np.random.default_rng(1)

mu = population.mean()
sample = rng.choice(population, size=5, replace=True)

centers = np.linspace(mu - 12, mu + 12, 400)
ssd = [(np.sum((sample - c)**2)) for c in centers]

plt.figure(figsize=(9, 4.5))
plt.plot(centers, ssd, color="tab:blue", linewidth=2)
plt.axvline(sample.mean(), color="tab:green", linestyle="--", linewidth=2,
            label=f"sample mean {sample.mean():.1f} (the minimizer)")
plt.axvline(mu, color="tab:red", linestyle=":", linewidth=2,
            label=f"population mean {mu:.1f}")
plt.title("Sum of squared deviations vs. the center you measure from")
plt.xlabel("center c")
plt.ylabel("$\\sum (x_i - c)^2$")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("The parabola bottoms out exactly at x_bar: deviations measured from the")
print("sample's own mean are as small as they can be -- systematically smaller")
print("than deviations from mu. Hence the naive estimator undershoots.")

## 5. Visualizations

### The two datasets and their four variance numbers

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 4.6), sharex=True)
for ax, d, name in [(axes[0], dataset_a, "A"), (axes[1], dataset_b, "B")]:
    ax.axhline(0, color="black", linewidth=1)
    ax.plot(d, np.zeros_like(d), "o", markersize=14, color="tab:blue", zorder=3)
    ax.axvline(160, color="tab:red", linestyle="--", linewidth=1.5)
    ax.set_yticks([])
    ax.set_title(f"dataset {name}: naive = {np.var(d):.1f},   s² = {np.var(d, ddof=1):.3f}",
                 fontsize=11)
    ax.grid(True, alpha=0.3, axis="x")
axes[1].set_xlabel("height (cm)")
plt.suptitle("Same mean (160), different spreads -- s² is always the larger estimate")
plt.tight_layout()
plt.show()

### The bias curve: naive vs s² across sample sizes

In [ ]:
rng = np.random.default_rng(2)

ns = np.arange(2, 41)
avg_naive, avg_s2 = [], []
for n in ns:
    samples = population[rng.integers(0, len(population), size=(20_000, n))]
    avg_naive.append(samples.var(axis=1, ddof=0).mean())
    avg_s2.append(samples.var(axis=1, ddof=1).mean())

plt.figure(figsize=(9.5, 5))
plt.plot(ns, avg_naive, "o-", color="tab:orange", label="average naive estimate (÷n)")
plt.plot(ns, avg_s2, "o-", color="tab:green", label="average s² estimate (÷(n-1))")
plt.plot(ns, population.var() * (ns - 1) / ns, ":", color="black",
         label=r"theory: $\sigma^2 (n-1)/n$")
plt.axhline(population.var(), color="tab:red", linestyle="--",
            label=r"true $\sigma^2$")
plt.title("The naive estimator's bias fades as n grows; s² is unbiased at every n")
plt.xlabel("sample size n")
plt.ylabel("average variance estimate")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### The distributions of the two estimators (n = 5)

Bias is about the *center* of an estimator's distribution: both estimators scatter widely at $n=5$, but the naive one's whole distribution sits shifted left of $\sigma^2$.

In [ ]:
rng = np.random.default_rng(3)

samples = population[rng.integers(0, len(population), size=(60_000, 5))]
naive = samples.var(axis=1, ddof=0)
s2 = samples.var(axis=1, ddof=1)

plt.figure(figsize=(10, 5))
plt.hist(naive, bins=80, density=True, alpha=0.5, color="tab:orange",
         label=f"naive (mean {naive.mean():.1f})")
plt.hist(s2, bins=80, density=True, alpha=0.5, color="tab:green",
         label=f"s² (mean {s2.mean():.1f})")
plt.axvline(population.var(), color="tab:red", linestyle="--", linewidth=2,
            label=f"true sigma² = {population.var():.1f}")
plt.title("Sampling distributions of the two estimators at n = 5")
plt.xlabel("variance estimate")
plt.ylabel("density")
plt.xlim(0, 250)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** For the sample $[4, 8, 6, 2]$, compute the sample mean, the naive variance estimate, and $s^2$ — by hand, then check with `np.var` and its `ddof` argument.

**Exercise 2.** A colleague reports "variance = 12.5" for a sample of 100 without saying which divisor they used. How large is the maximum possible ambiguity, and why is it not worth chasing?

### Intermediate

**Exercise 3.** Repeat the hat-game enumeration for samples of size $n = 3$ (27 outcomes): confirm the naive estimator's average is $\sigma^2 \cdot \frac{n-1}{n} = \frac23 \cdot \frac23 = \frac49$ and that $s^2$ again averages exactly $\frac23$.

**Exercise 4.** The other cheat is fine: if you *did* know $\mu$, the estimator $\frac1n\sum(x_i - \mu)^2$ (dividing by $n$!) is unbiased. Verify by simulation on the normal population, and explain why no correction is needed in that case.

### Advanced

**Exercise 5.** Prove the bias factor numerically: the identity $E\big[\tfrac1n\sum(x_i-\bar{x})^2\big] = \sigma^2\frac{n-1}{n}$ says exactly one $\sigma^2/n$ is lost — the variance of $\bar{x}$ itself. Verify by simulation that (naive estimate) + (variance of $\bar x$) averages to $\sigma^2$.

**Exercise 6.** $s^2$ is unbiased for the variance — but $s = \sqrt{s^2}$ is *not* unbiased for the standard deviation! Verify by simulation at $n = 3, 5, 10$ (average of $s$ vs. true $\sigma$). Which way is it biased, and what mathematical fact about square roots explains it?

## 7. Solutions

In [ ]:
# Exercise 1: by hand and by numpy
x = np.array([4, 8, 6, 2])
m = x.mean()
ssd = np.sum((x - m)**2)
print(f"Exercise 1: mean = {m}, squared deviations sum = {ssd}")
print(f"naive = {ssd}/4 = {ssd/4}   (np: {np.var(x)})")
print(f"s^2   = {ssd}/3 = {ssd/3:.3f} (np: {np.var(x, ddof=1):.3f})")

In [ ]:
# Exercise 2: divisor ambiguity at n = 100
low, high = 12.5, 12.5 * 100 / 99
print(f"Exercise 2: the two readings are {low} (if /n... then s^2 = {high:.3f})")
print(f"-- a {100 * (high/low - 1):.1f}% difference. At n = 100 the ambiguity is")
print("~1%, far below the estimate's own sampling noise: not worth chasing,")
print("exactly the practitioner's rule from the video.")

In [ ]:
# Exercise 3: the hat game with n = 3
from itertools import product
naive_list, s2_list = [], []
for draw in product([1, 2, 3], repeat=3):
    d = np.array(draw)
    naive_list.append(d.var(ddof=0))
    s2_list.append(d.var(ddof=1))
print(f"Exercise 3: {len(naive_list)} samples")
print(f"average naive = {np.mean(naive_list):.4f}   (theory 4/9 = {4/9:.4f})")
print(f"average s^2   = {np.mean(s2_list):.4f}   (= 2/3)")

In [ ]:
# Exercise 4: knowing mu removes the bias
rng = np.random.default_rng(4)

mu = population.mean()
sigma2 = population.var()
samples = population[rng.integers(0, len(population), size=(60_000, 5))]
known_mu = np.mean((samples - mu)**2, axis=1)
print(f"Exercise 4: average of (1/n) sum (x - mu)^2 = {known_mu.mean():.2f}"
      f"   (sigma^2 = {sigma2:.2f})")
print("Unbiased with divisor n! The bias came only from substituting x_bar,")
print("which is fitted to the sample; the true mu is a fixed target, and squared")
print("deviations from it average to sigma^2 with no shrinkage to correct.")

In [ ]:
# Exercise 5: the lost sigma^2/n is exactly Var(x_bar)
rng = np.random.default_rng(5)

for n in [2, 5, 10]:
    samples = population[rng.integers(0, len(population), size=(60_000, n))]
    naive = samples.var(axis=1, ddof=0)
    var_xbar = samples.mean(axis=1).var()
    print(f"n = {n:>2}: avg naive + Var(x_bar) = {naive.mean():.2f} + {var_xbar:.2f}"
          f" = {naive.mean() + var_xbar:.2f}   (sigma^2 = {sigma2:.2f})")
print("The cheat hides exactly the variability of x_bar itself -- sigma^2/n --")
print("and adding it back restores the target: the bias, quantified.")

In [ ]:
# Exercise 6: s is biased for sigma
rng = np.random.default_rng(6)

sigma = population.std()
print(f"Exercise 6: true sigma = {sigma:.3f}")
for n in [3, 5, 10]:
    samples = population[rng.integers(0, len(population), size=(60_000, n))]
    s = np.sqrt(samples.var(axis=1, ddof=1))
    print(f"n = {n:>2}: average of s = {s.mean():.3f}")
print("Biased LOW: the square root is concave, so E[sqrt(V)] < sqrt(E[V])")
print("(Jensen's inequality). Unbiasedness is not preserved by nonlinear")
print("transformations -- fixing the variance does not fix its square root.")

## 8. Conclusion

- The natural estimator — population formula with $\bar{x}$ substituted for $\mu$ — is $\widehat{\mathrm{Var}} = \frac1n\sum(x_i - \bar{x})^2$, and it is **biased low**: the sample mean is fitted to the sample, so deviations from it are systematically too small (it minimizes them!).
- The 1-2-3 hat game convicts it by exhaustive enumeration: population variance $\frac23$, average naive estimate over all 9 two-draw samples $\frac13$.
- The fix is the **sample variance** $s^2 = \frac{1}{n-1}\sum(x_i - \bar{x})^2$: same enumeration now averages exactly $\frac23$ — unbiased, in general.
- The video's datasets: $1.7 \to 2.125$ and $50.8 \to 63.5$ — $s^2$ always slightly larger.
- Perspective: the correction matters at small $n$ and vanishes at large $n$ (if it visibly matters, worry about your sample size, not your divisor); some methods like maximum likelihood legitimately divide by $n$; and unbiasedness is fragile — $s^2$ is unbiased but $s$ is not.

$s^2$ is the variance estimate used throughout the rest of the course. Next: the **law of large numbers** — why all these sample quantities converge to their population targets.